# Invoke a Foundry agent via REST - single-shot

Mirror of step 4 in [`08-01-create-versioned-storytelling-agent.ipynb`](../08-01-create-versioned-storytelling-agent.ipynb) (the `responses.create` call), executed directly with `requests` instead of the OpenAI SDK. Targets the `storytelling-agent` created in 08-01.

See [`08-09-00-invoke-agent-via-rest.md`](08-09-00-invoke-agent-via-rest.md) for the endpoint, auth, and body-shape contract this notebook implements.

## 1. Setup

Load `ALPHA_FOUNDRY_PROJECT_ENDPOINT` from the repo `.env` and derive the Responses URL: `{endpoint}/openai/v1/responses`. This is the same `base_url + /responses` that `project_client.get_openai_client()` configures internally.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

import requests
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

AGENT_NAME = 'storytelling-agent'

repo_root = Path(subprocess.run(
    'git rev-parse --show-toplevel', shell=True, capture_output=True, text=True
).stdout.strip())
load_dotenv(repo_root / '.env', override=True)

endpoint = os.environ['ALPHA_FOUNDRY_PROJECT_ENDPOINT'].rstrip('/')
responses_url = f'{endpoint}/openai/v1/responses'

print(f'Endpoint     : {endpoint}')
print(f'Responses URL: {responses_url}')
print(f'Agent name   : {AGENT_NAME}')

## 2. Get a bearer token

Audience is `https://ai.azure.com/.default` - the same scope the SDK uses (see `get_bearer_token_provider(credential, 'https://ai.azure.com/.default')` in `azure-ai-projects` `_patch.py`).

In [ ]:
credential = DefaultAzureCredential()
access_token = credential.get_token('https://ai.azure.com/.default').token
print(f'Token acquired (length: {len(access_token)} chars)')

## 3. Build the request body

The JSON body is exactly what the SDK assembles from the `input` argument plus `extra_body={'agent_reference': ...}`. `agent_reference.name` resolves to the latest version of the agent; add `'version': '<n>'` to pin a specific version.

In [ ]:
headers = {
    'Authorization': f'Bearer {access_token}',
    'Content-Type': 'application/json',
}

body = {
    'input': [
        {'role': 'user', 'content': 'Tell me a one line story about a curious robot.'}
    ],
    'agent_reference': {
        'name': AGENT_NAME,
        'type': 'agent_reference',
    },
}

print(json.dumps(body, indent=2))

## 4. POST to /responses

The single call that replaces `openai_client.responses.create(...)`.

In [ ]:
response = requests.post(responses_url, headers=headers, json=body, timeout=60)
response.raise_for_status()
result = response.json()

# The raw REST JSON has no `output_text` key. That field is a convenience the
# OpenAI SDK's typed Response synthesises by concatenating every output_text
# part. Over REST we aggregate it ourselves from the structured `output` array.
output_text = ''.join(
    part['text']
    for item in result.get('output', [])
    if item.get('type') == 'message'
    for part in item.get('content', [])
    if part.get('type') == 'output_text'
)

print(f'HTTP status : {response.status_code}')
print(f'Response id : {result["id"]}')
print(f'Status      : {result["status"]}')
print()
print('Output text:')
print(output_text)

## 5. Inspect the structured output

`output_text` is a convenience aggregate of all `output_text` parts across the output items. The structured `output` array is where richer item types (`function_call`, `mcp_tool_call`, `mcp_approval_request`, etc) appear in more complex agents.

In [ ]:
for i, item in enumerate(result.get('output', [])):
    print(f'output[{i}].type = {item.get("type")}')
    if item.get('type') == 'message':
        for j, part in enumerate(item.get('content', [])):
            print(f'  content[{j}].type = {part.get("type")}')
            if part.get('type') == 'output_text':
                print(f'  content[{j}].text = {part.get("text")}')

## Summary

| Step | SDK (`openai_client.responses.create`) | REST equivalent (this notebook) |
|------|----------------------------------------|---------------------------------|
| URL | implicit - `base_url = {endpoint}/openai/v1` | `f'{endpoint}/openai/v1/responses'` |
| Auth | `get_bearer_token_provider(credential, 'https://ai.azure.com/.default')` | `credential.get_token('https://ai.azure.com/.default').token` |
| Body | `input=...`, `extra_body={'agent_reference': ...}` | dict with `input` + `agent_reference` keys |
| Response | typed `Response` object - `.output_text`, `.output`, `.id` | JSON dict - `output`, `id` (no `output_text`; aggregate it yourself from `output`) |

Next: [`08-09-02-rest-multi-turn.ipynb`](08-09-02-rest-multi-turn.ipynb) continues a response chain using `previous_response_id`.</cell id="section-summary">